# 🏡 Aetheris ArchViz AI Studio — Google Colab GPU Server (T4 15GB Miễn Phí)
### ⚡ Tự Động Kết Nối 100% Về Web App (Zero-Configuration):
1. Chọn menu **Thời gian chạy (Runtime)** > **Thay đổi loại thời gian chạy (Change runtime type)** > Chọn **T4 GPU** > Bấm **Lưu**.
2. Bấm nút **Play ▶ (Chạy)** ở ô bên dưới.
3. Server sẽ tự động kích hoạt ComfyUI + Backend API và tự động đồng bộ link về Web App!

In [ ]:
#@title 🚀 1-CLICK RUN: Cài Đặt & Tự Động Kết Nối Web App Toàn Diện
import os, subprocess, time, re, urllib.request, json
from IPython.display import display, HTML, clear_output

GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN", "")
REPO_NAME = "Neito112/comfyui-archviz-studio"

print("⏳ [1/4] Đang cài đặt môi trường GPU & Aria2 siêu tốc...")
%cd /content
!apt-get update -qq && apt-get install -y -qq aria2 psmisc > /dev/null 2>&1

# 1. Clone hoặc Cập nhật Repository dự án
if not os.path.exists("/content/comfyui-archviz-studio"):
    print("⚡ Đang tải mã nguồn dự án Aetheris Studio...")
    !git clone https://github.com/Neito112/comfyui-archviz-studio.git /content/comfyui-archviz-studio > /dev/null 2>&1
else:
    %cd /content/comfyui-archviz-studio
    !git pull > /dev/null 2>&1

# 2. Clone ComfyUI Core nếu chưa có
if not os.path.exists("/content/ComfyUI"):
    print("⚡ Đang tải ComfyUI Engine...")
    !git clone --depth 1 https://github.com/comfyanonymous/ComfyUI /content/ComfyUI > /dev/null 2>&1

# 3. Clone Custom Nodes thiết yếu
%cd /content/ComfyUI/custom_nodes
if not os.path.exists("ComfyUI-Manager"):
    !git clone --depth 1 https://github.com/ltdrdata/ComfyUI-Manager.git > /dev/null 2>&1
if not os.path.exists("comfyui_controlnet_aux"):
    !git clone --depth 1 https://github.com/Fannovel16/comfyui_controlnet_aux.git > /dev/null 2>&1

# 4. Cài đặt Python Dependencies
print("📦 [2/4] Đang cài đặt PyTorch CUDA...")
%cd /content/ComfyUI
!pip install -q -r requirements.txt > /dev/null 2>&1
!pip install -q requests pillow > /dev/null 2>&1

# 5. Cài đặt Cloudflare Tunnel
if not os.path.exists("/usr/local/bin/cloudflared"):
    !wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
    !dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

# 6. Tải Đầy Đủ Cả 2 Dòng Model: SD 1.5 & SDXL Juggernaut & ControlNet Depth
print("📥 [3/4] Đang tải AI Models kiến trúc đa luồng (SD1.5, SDXL Juggernaut & ControlNet)...")
os.makedirs("/content/ComfyUI/models/checkpoints", exist_ok=True)
os.makedirs("/content/ComfyUI/models/controlnet", exist_ok=True)

# Realistic Vision V5.1 (SD1.5)
rv_path = "/content/ComfyUI/models/checkpoints/Realistic_Vision_V5.1.safetensors"
if not os.path.exists(rv_path):
    !aria2c -x 16 -s 16 -k 1M -c -d /content/ComfyUI/models/checkpoints -o Realistic_Vision_V5.1.safetensors https://huggingface.co/SG161222/Realistic_Vision_V5.1_noVAE/resolve/main/Realistic_Vision_V5.1_fp16-no-ema.safetensors > /dev/null 2>&1

# Juggernaut XL v9 (SDXL)
sdxl_path = "/content/ComfyUI/models/checkpoints/Juggernaut-XL_v9_RunDiffusionPhoto_v2.safetensors"
if not os.path.exists(sdxl_path):
    print("  🌟 Đang tải thêm model Juggernaut XL (6.7GB)...")
    !aria2c -x 16 -s 16 -k 1M -c -d /content/ComfyUI/models/checkpoints -o Juggernaut-XL_v9_RunDiffusionPhoto_v2.safetensors https://huggingface.co/RunDiffusion/Juggernaut-XL-v9/resolve/main/Juggernaut-XL_v9_RunDiffusionPhoto_v2.safetensors > /dev/null 2>&1

# ControlNet Depth
cn_path = "/content/ComfyUI/models/controlnet/control_v11f1p_sd15_depth.pth"
if not os.path.exists(cn_path):
    !aria2c -x 16 -s 16 -k 1M -c -d /content/ComfyUI/models/controlnet -o control_v11f1p_sd15_depth.pth https://huggingface.co/lllyasviel/ControlNet-v1-1/resolve/main/control_v11f1p_sd15_depth.pth > /dev/null 2>&1

# 7. Khởi chạy ComfyUI trên cổng 8188 & Backend API trên cổng 8000
print("⚡ [4/4] Đang khởi động ComfyUI & Backend API...")
!fuser -k 8188/tcp > /dev/null 2>&1
!fuser -k 8000/tcp > /dev/null 2>&1
!killall cloudflared > /dev/null 2>&1

os.system("nohup python /content/ComfyUI/main.py --port 8188 --listen 0.0.0.0 --highvram --dont-print-server > /tmp/comfy.log 2>&1 &")

# Chờ ComfyUI sẵn sàng
for _ in range(40):
    try:
        req = urllib.request.Request("http://127.0.0.1:8188/system_stats")
        with urllib.request.urlopen(req, timeout=1) as resp:
            if resp.status == 200:
                break
    except Exception:
        pass
    time.sleep(1)

# Khởi động Backend API của Studio trên cổng 8000
%cd /content/comfyui-archviz-studio
os.system("PORT=8000 nohup python backend/app.py > /tmp/backend.log 2>&1 &")
time.sleep(3)

# 8. Mở Cloudflare Tunnel chuyển tiếp cổng 8000 (Backend API Studio)
if os.path.exists("/tmp/tunnel.log"):
    os.remove("/tmp/tunnel.log")

os.system("nohup cloudflared tunnel --url http://127.0.0.1:8000 --logfile /tmp/tunnel.log > /dev/null 2>&1 &")

public_url = None
for i in range(30):
    time.sleep(1)
    if os.path.exists("/tmp/tunnel.log"):
        with open("/tmp/tunnel.log", "r") as f:
            content = f.read()
            match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", content)
            if match:
                public_url = match.group(0)
                break

clear_output(wait=True)
if public_url:
    # 9. Tự động đồng bộ link Colab mới nhất về GitHub Repo qua GitHub API
    try:
        print("🔄 Đang tự động đồng bộ link về GitHub Web App...")
        # Lấy SHA của backend/settings.json
        get_req = urllib.request.Request(f"https://api.github.com/repos/{REPO_NAME}/contents/backend/settings.json")
        get_req.add_header("Authorization", f"Bearer {GITHUB_TOKEN}")
        get_req.add_header("User-Agent", "Colab-Auto-Sync")
        with urllib.request.urlopen(get_req) as g_resp:
            f_info = json.loads(g_resp.read().decode())
            sha = f_info.get("sha")
        
        # Đẩy commit mới cập nhật remote_server_url
        import base64
        new_settings = {"remote_server_url": public_url, "engine_mode": "local", "arch_model": "realistic_vision"}
        new_content_b64 = base64.b64encode(json.dumps(new_settings, indent=2).encode()).decode()
        put_req = urllib.request.Request(f"https://api.github.com/repos/{REPO_NAME}/contents/backend/settings.json", method="PUT")
        put_req.add_header("Authorization", f"Bearer {GITHUB_TOKEN}")
        put_req.add_header("User-Agent", "Colab-Auto-Sync")
        put_req.add_header("Accept", "application/vnd.github+json")
        put_data = json.dumps({"message": "auto-sync: update active Colab GPU URL from Google Colab", "content": new_content_b64, "sha": sha}).encode()
        with urllib.request.urlopen(put_req, data=put_data) as p_resp:
            print("✅ Đã tự động cập nhật link Colab lên Web App!")
    except Exception as sync_e:
        print(f"⚠️ Ghi chú đồng bộ: {sync_e}")

    html_output = f"""
    <div style="background: linear-gradient(135deg, #020617, #0f172a); border: 2px solid #10b981; border-radius: 16px; padding: 24px; color: white; font-family: system-ui, sans-serif; box-shadow: 0 10px 30px rgba(0,0,0,0.8); max-width: 650px;">
        <div style="display:flex; align-items:center; gap:10px; margin-bottom:12px;">
            <span style="font-size:24px;">🎉</span>
            <h2 style="color: #10b981; margin: 0; font-size:20px;">COMFYUI & BACKEND GPU SẴN SÀNG!</h2>
        </div>
        <p style="font-size: 14px; color: #cbd5e1; margin-bottom: 8px;">Link kết nối GPU Cloud của bạn:</p>
        <div style="background: #000; border: 1.5px dashed #38bdf8; padding: 14px; border-radius: 10px; font-family: monospace; font-size: 16px; color: #38bdf8; font-weight: bold; word-break: break-all; margin-bottom: 16px;">
            {public_url}
        </div>
        <div style="margin-top: 15px;">
            <a href="https://neito112.github.io/comfyui-archviz-studio/" target="_blank" style="display: inline-block; background: #8b5cf6; color: white; padding: 10px 20px; border-radius: 8px; font-weight: bold; text-decoration: none;">👉 Mở Web App Trực Tuyến Ngay</a>
        </div>
    </div>
    """
    display(HTML(html_output))
    print(f"\n🔗 ACTIVE SERVER URL: {public_url}\n")
else:
    print("❌ Đang tạo đường hầm, vui lòng chạy lại ô này sau 5 giây!")
